In [2]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.linalg import cholesky

def isPD(B):
    try:
        np.linalg.cholesky(B)
        return True
    except np.linalg.LinAlgError:
        return False

def nearestPD(A):
    B = (A + A.T) / 2
    _, s, V = np.linalg.svd(B)
    H = V.T @ np.diag(s) @ V
    A2 = (B + H) / 2
    A3 = (A2 + A2.T) / 2
    if isPD(A3):
        return A3
    spacing = np.spacing(np.linalg.norm(A))
    I = np.eye(A.shape[0])
    k = 1
    while not isPD(A3):
        mineig = np.min(np.real(np.linalg.eigvals(A3)))
        A3 += I * (-mineig * k**2 + spacing)
        k += 1
    return A3
    
# First, assign Male and Female with a slight bias
def biased_gender_assignment(x, median):
    # Bias: if below median, 55% Male, 45% Female
    # if above or equal to median, 45% Male, 55% Female
    return "Male" if np.random.rand() < 0.55 else "Female"
    
        
def generate_categorical_attributes(df_cont):
    n = len(df_cont)
    df_cat = pd.DataFrame(index=range(n))
    
    median_age = df_cont['Age'].median()
    #df_cat['Gender'] = df_cont['Age'].apply(lambda x: "Other" if abs(x - median_age) < 0.0005 else ("Male" if x < median_age else "Female"))
    # df_cat['Gender'] = df_cont['Age'].apply(lambda x: "Male" if x < median_age else "Female")

    # Randomly select 1% of entries to be "Other"
    # num_other = int(0.025 * len(df_cat))
    # other_indices = np.random.choice(df_cat.index, size=num_other, replace=False)
    # df_cat.loc[other_indices, 'Gender'] = 'Other'

    median_age = df_cont['Age'].median()
    df_cat['Gender'] = df_cont['Age'].apply(lambda x: biased_gender_assignment(x, median_age))

    # Then add a small percent as "Other"
    num_other = int(0.025 * len(df_cat))
    other_indices = np.random.choice(df_cat.index, size=num_other, replace=False)
    df_cat.loc[other_indices, 'Gender'] = 'Other'
    
    def learning_style(hours):
        if hours < 15:
            return "Auditory"
        elif hours < 25:
            return "Reading/Writing"
        elif hours < 35:
            return "Visual"
        else:
            return "Kinesthetic"

    def determine_use_of_educational_tech(row, df_cat, df_cont):
        # Higher use of educational tech is assumed for those who:
        # - Have completed many online courses
        # - Spend significant time studying
        # - Participate in discussions
        # - Attend classes regularly
        # - Have good final grades
        # - Have longer commute times (indicating more time for tech use)
        
        # Checking conditions based on both df_cat and df_cont attributes
        
        # Online Courses Completed or Study Hours per Week
        if df_cont.loc[row.name, 'Online_Courses_Completed'] >= 10 or df_cont.loc[row.name, 'Study_Hours_per_Week'] > 30:
            return "Yes"
        
        # Participation in Discussions and Attendance Rate
        elif df_cat.loc[row.name, 'Participation_in_Discussions'] == "Yes" and df_cont.loc[row.name, 'Attendance_Rate'] > 80:
            return "Yes"
        
        # Final Grade or Commute Time
        elif df_cont.loc[row.name, 'Final_Grade'] > 85 or df_cont.loc[row.name, 'Commute_Time'] > 60:
            return "Yes"
        
        # Default if none of the conditions are met
        else:
            return "No"

    df_cat['Preferred_Learning_Style'] = df_cont['Study_Hours_per_Week'].apply(learning_style)
    df_cat['Participation_in_Discussions'] = df_cont['Assignment_Completion_Rate'].apply(lambda x: "Yes" if x >= 75 else "No")
    df_cat['Use_of_Educational_Tech'] = df_cat.apply(lambda row: determine_use_of_educational_tech(row, df_cat, df_cont), axis=1)
    df_cat['Peer_Study_Group'] = ((df_cont['Attendance_Rate'] >= 80) & (df_cont['Study_Hours_per_Week'] >= 25))

    df_merge = df_cont.merge(df_cat, left_index=True, right_index=True)

    def stress_level(row):
        """
        Compute a stress level using multiple factors, with weights informed by research on academic stress.
        
        Factors and their weights:
          - Age & Use_of_Educational_Tech: 
                Older students (Age > 24) who use educational tech often adapt better to digital learning (-1.5 total).
          - Study_Hours_per_Week:
                Excessive study (>30 hours/week) increases stress (+1.5); very low hours (<15) slightly reduce it (-0.5).
          - Assignment_Completion_Rate:
                High completion (>=85%) suggests effective management (-1.0), while low rates (<70%) may increase stress (+1.5).
          - Online_Courses_Completed:
                A high number (>12) adds stress (+1.0); a very low number (<5) can slightly reduce it (-0.5).
          - Peer_Study_Group:
                Participation reduces stress (-1.0); absence raises it (+0.5).
          - Participation_in_Discussions:
                Active participation lowers stress (-1.0); non-participation adds (+0.5).
          - Preferred_Learning_Style:
                “Reading/Writing” is slightly favorable (-0.5) and “Auditory” might be slightly disadvantageous (+0.5); other styles are neutral.
        
        The cumulative score is then mapped to:
          - Score <= -1.5  -> "Low"
          - Score between -1.5 and 2.0 -> "Medium"
          - Score >= 2.0  -> "High"
        """
        score = 0.0
    
        # Factor 1: Age and Use_of_Educational_Tech
        if row['Age'] > 24:
            score -= 1.0
            if row['Use_of_Educational_Tech'] == "Yes":
                score -= 0.5
    
        # Factor 2: Study Hours per Week
        if row['Study_Hours_per_Week'] > 30:
            score += 1.5
        elif row['Study_Hours_per_Week'] < 15:
            score -= 0.5
    
        # Factor 3: Assignment Completion Rate
        if row['Assignment_Completion_Rate'] >= 85:
            score -= 1.0
        elif row['Assignment_Completion_Rate'] < 70:
            score += 1.5
    
        # Factor 4: Online Courses Completed
        if row['Online_Courses_Completed'] > 12:
            score += 1.0
        elif row['Online_Courses_Completed'] < 5:
            score -= 0.5
    
        # Factor 5: Peer Study Group participation
        if row['Peer_Study_Group']:
            score -= 1.0
        else:
            score += 0.5
    
        # Factor 6: Participation in Discussions
        if row['Participation_in_Discussions'] == "Yes":
            score -= 1.0
        else:
            score += 0.5
    
        # Factor 7: Preferred Learning Style adjustments
        learning_style_adjust = {
            "Reading/Writing": -0.5,
            "Visual": 0,
            "Auditory": 0.5,
            "Kinesthetic": 0
        }
        score += learning_style_adjust.get(row['Preferred_Learning_Style'], 0)
    
        # Map the total score to stress categories
        if score <= -1.5:
            return "Low"
        elif score >= 2.0:
            return "High"
        else:
            return "Medium"

    df_cat['Self_Reported_Stress_Level'] = df_merge.apply(stress_level, axis=1)
    

    def parental_education(income):
        if income < 70000:
            return "High_School_Diploma"
        elif income < 100000:
            return "Some_College_No_Degree"
        elif income < 130000:
            return "Associate_Degree"
        elif income < 160000:
            return "Bachelors_Degree"
        elif income < 190000:
            return "Masters_Degree"
        else:
            return "Doctorate"

    df_cat['Parental_Education'] = df_cont['Parental_Income'].apply(parental_education)

    def health_metrics(sleep):
        if sleep > 8:
            return "Excellent"
        elif sleep >= 6:
            return "Average"
        else:
            return "Poor"

    df_cat['Health_Metrics'] = df_cont['Sleep_Hours_per_Night'].apply(health_metrics)
    return df_cat

def generate_continuous_variables(n):
    var_names = ['Age', 'Study_Hours', 'Online_Courses', 'Assignment_Rate',
                 'Exam_Score', 'Attendance', 'Sleep_Hours', 'Final_Grade',
                 'Parental_Income', 'Extracurricular_Activities', 'Commute_Time', 'Job_Hours']

    params = {
        'Age': (18, 29, 22, 2),
        'Study_Hours': (5, 49, 25, 8),
        'Online_Courses': (0, 20, 10, 3),
        'Assignment_Rate': (50, 100, 75, 8),
        'Exam_Score': (40, 100, 70, 10),
        'Attendance': (50, 100, 80, 10),
        'Sleep_Hours': (4, 10, 7.5, 1),
        'Final_Grade': (70, 95, 82, 5),
        'Parental_Income': (19800, 229956, 100000, 30000),
        'Extracurricular_Activities': (9, 24, 16, 2),
        'Commute_Time': (10, 178, 90, 30),
        'Job_Hours': (-0.03, 32, 12, 4)
    }

    d = len(var_names)
    target_means = np.array([params[v][2] for v in var_names])
    target_stds = np.array([params[v][3] for v in var_names])

    corr = np.array([
        [ 1.0,  -0.20, -0.30,  0.30,  0.30,  0.20, -0.30,  0.20,  0.30,  0.00,  0.20,  0.30],
        [-0.20,  1.0,   0.50,  0.70,  0.80,  0.60, -0.30,  0.80,  0.00, -0.20, -0.30, -0.40],
        [-0.30,  0.50,  1.0,   0.40,  0.30,  0.30, -0.20,  0.40,  0.20,  0.00, -0.20, -0.30],
        [ 0.30,  0.70,  0.40,  1.0,   0.70,  0.80, -0.20,  0.80,  0.20,  0.00, -0.30, -0.30],
        [ 0.30,  0.80,  0.30,  0.70,  1.0,   0.80, -0.30,  0.90,  0.30,  0.20, -0.40, -0.40],
        [ 0.20,  0.60,  0.30,  0.80,  0.80,  1.0,   -0.20,  0.90,  0.30,  0.20, -0.50, -0.30],
        [-0.30, -0.30, -0.20, -0.20, -0.30, -0.20,  1.0,   -0.20,  0.0,   0.30,  0.0,  -0.20],
        [ 0.20,  0.80,  0.40,  0.80,  0.90,  0.90, -0.20,  1.0,   0.40,  0.30, -0.50, -0.60],
        [ 0.30,  0.00,  0.20,  0.20,  0.30,  0.30,  0.0,   0.40,  1.0,   0.30,  0.20,  0.0 ],
        [ 0.00, -0.20,  0.00,  0.00,  0.20,  0.20,  0.30,  0.30,  0.30,  1.0,   -0.20, -0.30],
        [ 0.20, -0.30, -0.20, -0.30, -0.40, -0.50,  0.0,   -0.50,  0.20, -0.20,  1.0,   0.20],
        [ 0.30, -0.40, -0.30, -0.30, -0.40, -0.30, -0.20,  -0.60,  0.0,  -0.30,  0.20,  1.0]
    ])

    corr = nearestPD(corr)
    Z = np.random.multivariate_normal(np.zeros(d), corr, n)
    U = norm.cdf(Z)

    X = np.zeros_like(U)
    for i, var in enumerate(var_names):
        X[:, i] = norm.ppf(U[:, i], loc=target_means[i], scale=target_stds[i])
        low, high = params[var][0], params[var][1]
        X[:, i] = np.clip(X[:, i], low, high)

    df_cont = pd.DataFrame(X, columns=var_names)
    df_cont.rename(columns={
        'Study_Hours': 'Study_Hours_per_Week',
        'Online_Courses': 'Online_Courses_Completed',
        'Assignment_Rate': 'Assignment_Completion_Rate',
        'Attendance': 'Attendance_Rate',
        'Sleep_Hours': 'Sleep_Hours_per_Night',
        'Job_Hours': 'Part_Time_Job_Hours'
    }, inplace=True)

    df_cont['Exam_Score'] += 2 * np.sin(df_cont['Study_Hours_per_Week'] * np.pi / 30)
    df_cont['Assignment_Completion_Rate'] += 2 * np.cos(df_cont['Attendance_Rate'] * np.pi / 50)
    df_cont['Attendance_Rate'] += 2 * np.sin(df_cont['Online_Courses_Completed'] * np.pi / 20)

    for col in ['Assignment_Completion_Rate', 'Exam_Score', 'Attendance_Rate', 'Final_Grade']:
        df_cont[col] = df_cont[col].round(2)

    for col in ['Study_Hours_per_Week', 'Sleep_Hours_per_Night', 'Commute_Time', 'Part_Time_Job_Hours']:
        df_cont[col] = df_cont[col].round(1)

    df_cont['Age'] = df_cont['Age'].round().astype(int)
    df_cont['Parental_Income'] = df_cont['Parental_Income'].round(2)
    df_cont['Extracurricular_Activities'] = df_cont['Extracurricular_Activities'].round(2)
    df_cont['Online_Courses_Completed'] = df_cont['Online_Courses_Completed'].round(2)

    return df_cont


def generate_social_media_time(df_cat, df_cont):
    """
    Generate daily social media usage (in hours) ensuring that the sum of sleep, study, and social media time
    does not exceed 24 hours. Adjustments are made based on:
      - Sleep_Hours_per_Night (from df_cont): Less than 6 hrs increases usage; more than 8 hrs decreases usage.
      - Commute_Time (from df_cont, in minutes): Longer commutes add to usage.
      - Study_Hours_per_Week (from df_cont): Used to compute daily study time.
      - Self_Reported_Stress_Level (from df_cat): Higher stress increases usage.
      - Health_Metrics (from df_cat): Poor health increases usage.
      
    Returns:
        A NumPy array with the computed daily social media usage in hours,
        rounded to two decimals and ensuring that social media time does not exceed (24 - sleep - study_daily).
    """
    import numpy as np
    n = len(df_cat)
    social = np.zeros(n)
    baseline = 3.0  # baseline social media usage in hours per day
    
    for i in range(n):
        # Continuous variables from df_cont
        sleep = df_cont.loc[i, 'Sleep_Hours_per_Night']         # in hours per day
        commute = df_cont.loc[i, 'Commute_Time']                  # in minutes per day
        study_weekly = df_cont.loc[i, 'Study_Hours_per_Week']
        study_daily = study_weekly / 7.0
        
        # Categorical variables from df_cat
        stress = df_cat.loc[i, 'Self_Reported_Stress_Level']
        health = df_cat.loc[i, 'Health_Metrics']
        
        # Start with the baseline usage
        sm_time = baseline

        # Adjust based on sleep: 
        #   Less than 6 hours sleep may increase social media usage;
        #   More than 8 hours sleep tends to reduce it.
        if sleep < 6:
            sm_time += 0.5
        elif sleep > 8:
            sm_time -= 0.5
            
        # Adjust for commute time (convert minutes to a rough hourly influence):
        if commute > 120:
            sm_time += 1.0
        elif commute > 60:
            sm_time += 0.5
        
        # Adjust based on stress level:
        if stress == 'High':
            sm_time += 1.0
        elif stress == 'Medium':
            sm_time += 0.5
        elif stress == 'Low':
            sm_time -= 0.5
        
        # Adjust based on health metrics:
        if health == "Poor":
            sm_time += 0.5
        elif health == "Excellent":
            sm_time -= 0.5
            
        # Add some small random noise (mean 0, std 0.25 hours)
        sm_time += np.random.normal(0, 0.25)
        
        # Calculate maximum available time per day for social media
        max_available = 24.0 - (sleep + study_daily)
        # Ensure the generated social media time is not above the maximum available time
        sm_time = np.clip(sm_time, 0, max_available)
        social[i] = sm_time
    
    return np.round(social, 2)



def convert_to_letter_grade(score):
    if score >= 93:
        return 'A'
    elif score >= 90:
        return 'A-'
    elif score >= 87:
        return 'B+'
    elif score >= 83:
        return 'B'
    elif score >= 80:
        return 'B-'
    elif score >= 77:
        return 'C+'
    elif score >= 73:
        return 'C'
    elif score >= 70:
        return 'C-'
    elif score >= 60:
        return 'D'
    else:
        return 'F'


def combine_data(n=10000):
    df_cont = generate_continuous_variables(n)
    df_cat = generate_categorical_attributes(df_cont)

    # Generate and insert only once into df_cont (not both)
    df_cont['Time_Spent_on_Social_Media'] = generate_social_media_time(df_cat, df_cont)

    # Remove Time_Spent_on_Social_Media from df_cat if it was already present
    if 'Time_Spent_on_Social_Media' in df_cat.columns:
        df_cat.drop(columns=['Time_Spent_on_Social_Media'], inplace=True)

    df = pd.concat([df_cont, df_cat], axis=1)
    df.insert(0, 'Student_ID', [f"S{i+1:05d}" for i in range(n)])

    edu_boost = {
        "Doctorate": 4,
        "Masters_Degree": 3,
        "Bachelors_Degree": 2.5,
        "Associate_Degree": 1.5,
        "Some_College_No_Degree": 0.5,
        "High_School_Diploma": 0
    }

    final_grade = df['Final_Grade'].astype(float).copy()
    final_grade += df['Parental_Education'].map(edu_boost)
    final_grade += np.minimum(5, ((df['Parental_Income'] - 50000) // 25000))
    final_grade += np.where((df['Extracurricular_Activities'] >= 12) & (df['Extracurricular_Activities'] <= 18), 1, 0)
    final_grade += np.where(df['Commute_Time'] > 120, np.where(df['Attendance_Rate'] < 80, -5, -2), 0)
    df['Final_Grade'] = final_grade.round(2)
    df['Letter_Grade'] = df['Final_Grade'].apply(convert_to_letter_grade)


    at_risk_grades = ['B-', 'C+', 'C', 'C-', 'D', 'F']
    df['AtRisk'] = np.where(df['Letter_Grade'].isin(at_risk_grades), 'Yes', 'No')

    cols = [col for col in df.columns if col not in ['Final_Grade', 'Letter_Grade', 'AtRisk']]
    cols += ['Final_Grade', 'Letter_Grade', 'AtRisk']
    df = df[cols]

    return df


# Example usage
if __name__ == "__main__":
    df = combine_data(20000)
    df.to_csv("data/realistic_student_data.csv", index=False)
    print("Dataset generated ...")

Dataset generated ...
